In [1]:
library(tidyverse)
library(tximport)
library(DESeq2)
library(rtracklayer)
library(ggrepel)
library(RColorBrewer)
library(pheatmap)
library(patchwork)
library(ggrastr)
library(gridExtra)
library(here)
here::i_am("figures/Figure4/DEGs_stim_genes.ipynb")

theme_set(theme_bw() + theme(panel.grid=element_blank(), text=element_text(size=15)))

sessionInfo()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’

R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] here_1.0.2                  gridExtra_2.3              
 [3] ggrastr_1.0.2               patchwork_1.3.0            
 [5] pheatmap_1.0.12             RColorBrewer_1.1-3         
 [7] ggrepel_0.9.6               rtracklayer_1.58.0         
 [9] DESeq2_1.38.3               Sum

In [5]:
#GENCODE annotations
gtf.file <- here("inputs","GENCODE","gencode.v47.annotation.gtf.gz")
gencode <- import(gtf.file)
gencode <- gencode[which(gencode$type=="gene")]
#read in TSS coordinates w/ genes
gencode_tss <- read.delim(here("inputs", "GENCODE","gencode.v47.annotation.transcripts_tss.1kb.bed"), header=F, sep='\t')

In [6]:
#generate rowdata
row.data <- data.frame(gencode@elementMetadata)
rownames(row.data) <- row.data$gene_id
row.data$gene_id <- NULL
#generate column with cleaned ENSEMBL IDs
row.data$gene_id_base <- gsub("\\..*$", "", rownames(row.data)) 
#remove NA columns
row.data <- row.data[,colSums(is.na(row.data))<nrow(row.data)]
# add genome coordinates
row.data$chr <- as.character(gencode@seqnames)
row.data$start <- as.numeric(gencode@ranges@start)
row.data$end <- row.data$start + as.numeric(gencode@ranges@width)

row.data <- row.data %>% filter(gene_type=="protein_coding")

In [7]:
#read in TPM and counts data
path <- here("inputs","RSEM","GM12878","stimulus_bafinh")
files <- list.files(path, pattern="results", full.name=TRUE)
txi <- tximport(files,type="rsem", txIn=FALSE)

reading in files with read_tsv

1 
2 
3 
4 
5 
6 
7 
8 
9 
10 
11 
12 
13 
14 
15 
16 
17 
18 
19 
20 
21 
22 
23 
24 




In [8]:
#make column metadata
parse_file <- function(filename) {
  parts <- str_split(filename, "_", simplify = TRUE)

  # find the column that matches r<digit>
  rep_idx <- which(str_detect(parts, "^r\\d+$"))

  timepoint <- parts[2]                     # T1
  replicate <- parts[rep_idx]               # r1, r2, ...
  treatment_vec <- parts[3:(rep_idx - 1)]   # BRM or BRM_Dex
  treatment <- paste(treatment_vec, collapse = "_")

  tibble(
    filename = filename,
    timepoint = timepoint,
    treatment = treatment,
    replicate = replicate
  )
}

files <- list.files(path, pattern="results")
col.data <- do.call(rbind, lapply(files, parse_file)) %>% mutate(group=paste0(treatment,"_", timepoint))
col.data

filename,timepoint,treatment,replicate,group
<chr>,<chr>,<chr>,<chr>,<chr>
rep1_T1_BRM_Dex_r1_anno_rsem.genes.results,T1,BRM_Dex,r1,BRM_Dex_T1
rep1_T1_BRM_Dex_r2_anno_rsem.genes.results,T1,BRM_Dex,r2,BRM_Dex_T1
rep1_T1_BRM_IFN_r1_anno_rsem.genes.results,T1,BRM_IFN,r1,BRM_IFN_T1
rep1_T1_BRM_IFN_r2_anno_rsem.genes.results,T1,BRM_IFN,r2,BRM_IFN_T1
rep1_T1_BRM_r1_anno_rsem.genes.results,T1,BRM,r1,BRM_T1
rep1_T1_BRM_r2_anno_rsem.genes.results,T1,BRM,r2,BRM_T1
rep1_T1_Dex_r1_anno_rsem.genes.results,T1,Dex,r1,Dex_T1
rep1_T1_Dex_r2_anno_rsem.genes.results,T1,Dex,r2,Dex_T1
rep1_T1_IFN_r1_anno_rsem.genes.results,T1,IFN,r1,IFN_T1


In [9]:
#filter txi object
nrow(txi$abundance)
#first, filter to exclude non-protein coding genes using the row metadata
txi$abundance <- txi$abundance[match(row.data$gene_id_base, gsub("\\..*$", "", rownames(txi$abundance))),] 
txi$counts <- txi$counts[match(row.data$gene_id_base, gsub("\\..*$", "", rownames(txi$counts))),]
txi$length <- txi$length[match(row.data$gene_id_base, gsub("\\..*$", "", rownames(txi$length))),]
nrow(txi$abundance)
#next, filter to remove lowly-expressed and 0-length genes
filt <- which(apply(txi$abundance,MARGIN=1,FUN=max) <= 1 | #genes with low expression (where max TPM for all samples is <=1)
                apply(txi$length, MARGIN=1,FUN=min)==0) #genes with weird annotations (length=0)

#filter out these genes from txi object
txi$abundance <- txi$abundance[-filt,]
txi$counts <- txi$counts[-filt,]
txi$length <- txi$length[-filt,]

#get rid of any NA rows
txi$counts <- txi$counts[-which(!rowSums(is.finite(txi$abundance))),]
txi$length <- txi$length[-which(!rowSums(is.finite(txi$abundance))),]
txi$abundance <- txi$abundance[-which(!rowSums(is.finite(txi$abundance))),]
nrow(txi$abundance)
     
#filter row metadata to match rows of filtered txi object
row.data <- row.data[match(gsub("\\..*$", "", 
                                rownames(txi$abundance)), 
                           row.data$gene_id_base),]
nrow(txi$abundance)
nrow(row.data)

[1] 59526

[1] 20092

[1] 12190

[1] 12190

[1] 12190

In [67]:
deg_res <- function (des, row.data, contrast) 
{
    res <- results(des, contrast = contrast)
    res <- res %>% data.frame() %>% rownames_to_column("ENSEMBL_ID") %>% 
            mutate(gene=row.data$gene_name,change = case_when(log2FoldChange > log2(1.5) & 
                                      padj < 0.05 ~ "up", 
                                      log2FoldChange < -log2(1.5) & padj < 0.05 ~ "down", 
                                      TRUE ~ "ns"))
    return(res)
}

dds <- DESeqDataSetFromTximport(txi,col.data, design=~group)
des <- DESeq(dds)

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
using counts and average transcript lengths from tximport

estimating size factors

using 'avgTxLength' from assays(dds), correcting for library size

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [183]:
ifn_1h <- deg_res(des, row.data, c("group", "IFN_T1", "Veh_T1"))
ifn_6h <- deg_res(des, row.data, c("group", "IFN_T6", "Veh_T6"))
dex_1h <- deg_res(des, row.data, c("group", "Dex_T1", "Veh_T1"))
dex_6h <- deg_res(des, row.data, c("group", "Dex_T6", "Veh_T6"))

brm_ifn_1h <- deg_res(des, row.data, c("group", "BRM_IFN_T1", "IFN_T1"))
brm_dex_1h <- deg_res(des, row.data, c("group", "BRM_Dex_T1", "Dex_T1"))
brm_ifn_6h <- deg_res(des, row.data, c("group", "BRM_IFN_T6", "IFN_T6"))
brm_dex_6h <- deg_res(des, row.data, c("group", "BRM_Dex_T6", "Dex_T6"))

brm_ifn_veh_1h <- deg_res(des, row.data, c("group", "BRM_IFN_T1", "Veh_T1"))
brm_dex_veh_1h <- deg_res(des, row.data, c("group", "BRM_Dex_T1", "Veh_T1"))
brm_ifn_veh_6h <- deg_res(des, row.data, c("group", "BRM_IFN_T6", "Veh_T6"))
brm_dex_veh_6h <- deg_res(des, row.data, c("group", "BRM_Dex_T6", "Veh_T6"))


brm_1h <- deg_res(des, row.data, c("group", "BRM_T1", "Veh_T1"))
brm_6h <- deg_res(des, row.data, c("group", "BRM_T6", "Veh_T6"))

In [9]:
ifn_up <- which(ifn_1h$change=="up" | ifn_6h$change=="up")
dex_up <- which(dex_1h$change=="up" | dex_6h$change=="up")
ifn_down <- which(ifn_1h$change=="down" | ifn_6h$change=="down")
dex_down <- which(dex_1h$change=="down" | dex_6h$change=="down")

In [167]:
volcano <- function(table, rowdat, q_thresh=0.05, fc_thresh=log2(1.5), main="", top_n=10){
    temp <- table %>% mutate(gene=rowdat$gene_name) %>% 
            na.omit() %>%
            mutate(change = case_when(log2FoldChange > fc_thresh & 
                                      padj < q_thresh ~ "up", 
                                      log2FoldChange < -fc_thresh & padj < q_thresh ~ "down", 
                                      TRUE ~ "ns"))
    xmax <- max(abs(temp$log2FoldChange))
    n_down <- table(temp$change)[1]
    n_up <- table(temp$change)[3]

    #top10_up <- temp[temp$change=="up",] %>% arrange(desc(log2FoldChange)) %>% head(top_n)
    #top10_down <- temp[temp$change=="down",] %>% arrange(log2FoldChange) %>% head(top_n)
    top10s <- temp[temp$change == "up",] %>% arrange(desc(abs(log2FoldChange))) %>% head(top_n)

    ggplot(temp, aes(color=change, x=log2FoldChange,y=-log10(padj))) +
        geom_point(size=.5) +
        geom_vline(xintercept=c(-fc_thresh, fc_thresh), linetype=2) + 
        geom_hline(yintercept=-log10(q_thresh), linetype=2) + 
        xlim(-xmax, xmax) +
        scale_color_manual(values=c("down"=brewer.pal(3,"Paired")[2], "ns"="grey", "up"=brewer.pal(6,"Paired")[6])) +
        geom_text_repel(data=top10s, aes(label=gene), size=2, min.segment.length = 0, max.overlaps=100) +
        ggtitle(main, subtitle=paste("Down:", n_down, "; Up:", n_up)) 
}

hm_scale <- function(tpms,coldata,groups, scale="none", breaks=NA, colors=c("white", "red")){
    temp <- tpms[,coldata$group %in% groups] %>% +1 %>% log2()
    temp_coldata <- coldata[coldata$group %in% groups,]
    max <- max(temp) + 1
    
    d <- dist(scale(temp))
    h <- hclust(d)
    
    pheatmap(temp[h$order,], 
             color=colorRampPalette(colors)(100), 
             cluster_rows=if (scale=="row") TRUE else FALSE, 
             cluster_cols=FALSE, 
             border_col=NA, 
             labels_col=temp_coldata$group,
             show_rownames=FALSE,
             scale=scale,
             breaks= breaks,
             cellheight=1, 
             main=paste("n=", nrow(temp)))
}

In [71]:
ifn_volcanos <- (volcano(ifn_1h, row.data) / volcano(ifn_6h, row.data))
dex_volcanos <- (volcano(dex_1h, row.data) / volcano(dex_6h, row.data))

In [72]:
#one option - volcano plot (issue imo is that it doesn't capture similarity between responses)
volcano <- ifn_volcanos | dex_volcanos
ggsave("panelc_volcano.pdf", volcano, width=9)

Saving 9 x 7 in image


In [2]:
#write out DEGs
dir <- here("inputs", "DEGs", "GM12878",  "BAFinh_stimulus")
write.table(ifn_1h, paste0(dir,"IFNy_1h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(ifn_6h, paste0(dir,"IFNy_6h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(dex_1h, paste0(dir,"Dex_1h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(dex_6h, paste0(dir,"Dex_6h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_1h, paste0(dir,"BRM_1h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_6h, paste0(dir,"BRM_6h_DEGs.tsv"), sep='\t', row.names=FALSE)

write.table(brm_ifn_1h, paste0(dir,"BRM_IFNy_v_IFNy_1h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_dex_1h, paste0(dir,"BRM_Dex_v_Dex_1h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_ifn_6h, paste0(dir,"BRM_IFNy_v_IFNy_6h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_dex_6h, paste0(dir,"BRM_Dex_v_Dex_6h_DEGs.tsv"), sep='\t', row.names=FALSE)

write.table(brm_ifn_veh_1h, paste0(dir,"BRM_IFNy_v_Veh_1h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_dex_veh_1h, paste0(dir,"BRM_Dex_v_Veh_1h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_ifn_veh_6h, paste0(dir,"BRM_IFNy_v_Veh_6h_DEGs.tsv"), sep='\t', row.names=FALSE)
write.table(brm_dex_veh_6h, paste0(dir,"BRM_Dex_v_Veh_6h_DEGs.tsv"), sep='\t', row.names=FALSE)



ERROR: Error in parse(text = input): <text>:2:40: unexpected symbol
1: #write out DEGs
2: dir <- here("inputs", "DEGs", "GM12878".
                                          ^


In [10]:
#write gene metadata, sample metadata, and TPMs table
write.table(row.data, "IFNy_Dex_DEGs_rowdata.tsv", sep='\t', row.names=FALSE)
write.table(txi$abundance, "IFNy_Dex_DEGs_TPMs.tsv", sep='\t', row.names=FALSE)
write.table(col.data, "IFNy_Dex_DEGs_coldata.tsv", sep='\t', row.names=FALSE)